# 06 FTMO Vs Standard

Notebook này so sánh cùng một strategy dưới hai account mode:
- `standard`: nghiên cứu không bị giới hạn thực tế.
- `ftmo`: áp daily loss và max drawdown theo cấu hình FTMO.

Mục tiêu là thấy luật tài khoản làm thay đổi equity, trade count, return và drawdown như thế nào.


In [ ]:
# Cell 1 - Bootstrap đường dẫn import an toàn
#
# Notebook có thể được mở từ repo root, từ thư mục strategies/combo,
# hoặc từ một working directory khác trong VS Code/Jupyter. Vì vậy ta không
# dùng `config.py` làm marker root: trong strategies/combo cũng có config.py,
# rất dễ nhận nhầm thư mục strategy là repo root.
#
# Marker đáng tin cậy hơn là pyproject.toml + thư mục core_python/shared.
# Sau khi tìm được root thật, ta thêm cả repo root và core_python vào sys.path
# để import được `shared.*` và `strategies.combo.*`.

import sys
from pathlib import Path


def _find_root(start: Path, marker: str = 'pyproject.toml') -> Path:
    for p in [start, *start.parents]:
        if (p / marker).exists() and (p / 'core_python' / 'shared').exists():
            return p
    raise RuntimeError(f'Không tìm thấy repo root chứa {marker!r} và core_python/shared')


ROOT = _find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)
print('CORE =', CORE)


In [ ]:
# Cell 2 - Import thư viện và helper so sánh mode

from IPython.display import display

from core_python.strategies.combo.params import summary as strategy_summary
from core_python.strategies.combo.research_utils import (
    compare_metrics_frame,
    configure_notebook,
    plot_mode_comparison,
    show_ftmo_check,
    show_note,
    show_run_config,
)
from core_python.strategies.combo.portfolio.backtest import compare_account_modes
from core_python.strategies.combo.symbol.backtest import run_symbol_backtest

configure_notebook()
print(strategy_summary())


In [ ]:
# Cell 3 - Cấu hình so sánh account mode

RUN_CONFIG = {
    'symbol': 'US30',
    'portfolio_symbols': ['US30', 'US500', 'DE40', 'GOLD'],
    'initial_balance': 100_000.0,
    'date_from': '2023-01-01',
    'date_to': None,
    'max_bars': 30_000,
}

show_run_config('Cấu hình FTMO vs standard', RUN_CONFIG)


In [ ]:
# Cell 4 - So sánh ở cấp một symbol
#
# Chạy cùng symbol hai lần: standard và ftmo. Nếu FTMO có ít trade hoặc return thấp hơn,
# thường là do daily stop/max DD làm engine ngừng mở lệnh trong một số giai đoạn.

symbol_results = {}
for mode in ['standard', 'ftmo']:
    symbol_results[mode] = run_symbol_backtest(
        RUN_CONFIG['symbol'],
        init_eq=RUN_CONFIG['initial_balance'],
        account_mode=mode,
        date_from=RUN_CONFIG['date_from'],
        date_to=RUN_CONFIG['date_to'],
        max_bars=RUN_CONFIG['max_bars'],
    )

symbol_compare = compare_metrics_frame(symbol_results)
show_note('Symbol mode comparison', 'So sánh KPI của cùng một symbol dưới hai account mode.')
display(symbol_compare)


In [ ]:
# Cell 5 - So sánh ở cấp portfolio
#
# Portfolio comparison quan trọng hơn khi đánh giá challenge account, vì luật FTMO
# thường áp ở tổng tài khoản chứ không chỉ từng symbol riêng lẻ.

mode_results = compare_account_modes(
    symbol_keys=RUN_CONFIG['portfolio_symbols'],
    initial_balance=RUN_CONFIG['initial_balance'],
    date_from=RUN_CONFIG['date_from'],
    date_to=RUN_CONFIG['date_to'],
    max_bars=RUN_CONFIG['max_bars'],
)

portfolio_compare = compare_metrics_frame(mode_results)
show_note('Portfolio mode comparison', 'So sánh KPI portfolio giữa standard và ftmo.')
display(portfolio_compare)
show_ftmo_check(mode_results['ftmo'].metrics)


In [ ]:
# Cell 6 - Chart so sánh equity
#
# Đọc chart theo câu hỏi: FTMO có làm equity bớt rủi ro hơn không, hay chỉ làm mất cơ hội lợi nhuận?

plot_mode_comparison(symbol_results, mode_results)
